In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"ablation_A3_1s3w_hpo_v1"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: ablation_A3_1s3w_hpo_v1
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\ablation_A3_1s3w_hpo_v1.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'ablation_A3_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 7 trials.
Best value (Accuracy): 0.8084
Best Hyperparameters:
  outer_lr: 0.0001685216219363172
  wd: 4.299410126639929e-05
  maml_inner_steps: 15
  maml_alpha_init: 0.010379202670259113
  maml_alpha_init_eval: 0.004474796436994665
  maml_use_lslr: True
  use_lslr_at_eval: True
  use_maml_msl: False
  episodes_per_epoch_train: 100


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

#params_to_plot_ARCH = [
#    "cnn_base_filters", "lstm_hidden", "groupnorm_num_groups", "use_GlobalAvgPooling"
#]

# TODO: Why is label_smooth not here...
params_to_plot_OPT = [
    "outer_lr", "wd", #"label_smooth", #"ft_lr"
]

# TODO: Why is meta_batchsize not in here...
params_to_plot_MAML = [
    "maml_inner_steps", "maml_alpha_init_eval", "maml_use_lslr", "use_lslr_at_eval", "use_maml_msl", "episodes_per_epoch_train"
]

#params_to_plot_MOE_CORE = [
#    "num_experts", "MOE_top_k", "MOE_aux_coeff", "MOE_gate_temperature"
#]

#params_to_plot_MOE_AUX = [
#    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout"
#]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_MAML)
fig_slice.show()

In [12]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
2,2,0.808437,None,2026-04-20 09:17:12.133043,0 days 00:17:14.891615
3,3,0.765833,None,2026-04-20 09:35:10.130738,0 days 00:07:36.236622
4,4,0.700104,None,2026-04-20 09:43:29.198837,0 days 00:18:10.913553
0,0,0.681771,None,2026-04-20 09:00:17.006065,0 days 00:09:31.046181
1,1,0.646458,None,2026-04-20 09:10:36.473974,0 days 00:05:58.339166
5,5,NaN,None,2026-04-20 12:27:26.773788,NaT
6,6,NaN,None,2026-04-20 12:29:10.061961,NaT
